# One-vs-One (OvO) Strategy for Multi-Class Classification

This notebook demonstrates One-vs-One, a decomposition strategy that trains K(K-1)/2 binary classifiers for each pair of classes, with final prediction by majority voting.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.multiclass import OneVsOneClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import itertools

np.random.seed(42)

## 1. Load and Prepare Data

In [ ]:
iris = load_iris()
X, y = iris.data, iris.target
target_names = iris.target_names
n_classes = len(target_names)

# Calculate expected number of pairwise classifiers
n_pairwise = n_classes * (n_classes - 1) // 2

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

print(f"Number of classes: {n_classes}")
print(f"Expected pairwise classifiers: {n_pairwise}")
print(f"Class pairs: {list(itertools.combinations(range(n_classes), 2))}")

## 2. One-vs-One with SVM

In [ ]:
# OvO with SVM
ovo_svm = OneVsOneClassifier(SVC(kernel='rbf', random_state=42))
ovo_svm.fit(X_train, y_train)
y_pred_ovo = ovo_svm.predict(X_test)
acc_ovo = accuracy_score(y_test, y_pred_ovo)

print(f"One-vs-One SVM Accuracy: {acc_ovo:.4f}")
print(f"Number of pairwise classifiers created: {len(ovo_svm.estimators_)}")

## 3. OvO vs. OvR Comparison

In [ ]:
from sklearn.multiclass import OneVsRestClassifier

# OvR for comparison
ovr_svm = OneVsRestClassifier(SVC(kernel='rbf', random_state=42))
ovr_svm.fit(X_train, y_train)
y_pred_ovr = ovr_svm.predict(X_test)
acc_ovr = accuracy_score(y_test, y_pred_ovr)

print("\n=== OvO vs. OvR ===")
print(f"One-vs-One SVM: {acc_ovo:.4f} (pairwise classifiers: {len(ovo_svm.estimators_)})")
print(f"One-vs-Rest SVM: {acc_ovr:.4f} (binary classifiers: {len(ovr_svm.estimators_)})")

## 4. Analyze Pairwise Classifiers

In [ ]:
# Show pairwise class combinations
print("\nPairwise classifier breakdown:")
pairs = list(itertools.combinations(range(n_classes), 2))

for idx, (i, j) in enumerate(pairs):
    estimator = ovo_svm.estimators_[idx]
    
    # Create binary subproblem
    mask = (y_test == i) | (y_test == j)
    X_binary = X_test[mask]
    y_binary = (y_test[mask] == j).astype(int)  # j=1, i=0
    
    y_pred_binary = estimator.predict(X_binary)
    binary_acc = accuracy_score(y_binary, y_pred_binary)
    
    print(f"  Pair ({target_names[i]} vs. {target_names[j]}): {binary_acc:.4f}")

## 5. Strategy Comparison

In [ ]:
strategies = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42, multi_class='multinomial'),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'OvR SVM': OneVsRestClassifier(SVC(kernel='rbf', random_state=42)),
    'OvO SVM': OneVsOneClassifier(SVC(kernel='rbf', random_state=42))
}

print("\n=== MULTI-CLASS STRATEGY COMPARISON ===")
for name, model in strategies.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    print(f"{name:25s}: {acc:.4f}")

## 6. Confusion Matrix

In [ ]:
cm = confusion_matrix(y_test, y_pred_ovo)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=target_names, yticklabels=target_names)
plt.title('One-vs-One SVM Confusion Matrix')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.tight_layout()
plt.show()

## 7. Classification Report

In [ ]:
print("\nOne-vs-One SVM Classification Report:")
print(classification_report(y_test, y_pred_ovo, target_names=target_names))

## 8. OvO vs. OvR Trade-offs

| Aspect | One-vs-One | One-vs-Rest |
|--------|-----------|-------------|
| **Classifiers** | K(K-1)/2 | K |
| **Training data per classifier** | More balanced (2 classes) | Imbalanced (1 vs. K-1) |
| **Training time** | Slower (more models) | Faster (fewer models) |
| **Prediction time** | O(K²) decisions | O(K) decisions |
| **When to use** | K small, need balance | K large, speed critical |
| **Typical use** | SVM, KNN | Logistic Regression, tree |
| **For K=3** | 3 classifiers | 3 classifiers |
| **For K=10** | 45 classifiers | 10 classifiers |